In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# 1. Pre-Flight Verification Gate: Ensure all 4 independent files exist
required_files = ["weather_features.csv", "crowd_features.csv", "landslide_features.csv", "transport_features.csv"]
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    raise FileNotFoundError(f"Missing required upstream files: {missing_files}. Please run your previous pipelines first.")

In [3]:
# 2. Ingest all features layers
weather = pd.read_csv("weather_features.csv")
crowd = pd.read_csv("crowd_features.csv")
landslide = pd.read_csv("landslide_features.csv")
transport = pd.read_csv("transport_features.csv")

In [4]:
print("=====================================================================")
print("📊 STEP 2 VERIFICATION: INSPECTING ALL PIPELINE SHEET SHAPES")
print("=====================================================================")
print(f" -> Weather (Dynamic Timeline):   {weather.shape} (Expected: 18260 rows)")
print(f" -> Crowd (Dynamic Timeline):     {crowd.shape} (Expected: 18260 rows)")
print(f" -> Landslide (Static Dimension): {landslide.shape} (Expected: 10 rows)")
print(f" -> Transport (Static Dimension): {transport.shape} (Expected: 10 rows)")

📊 STEP 2 VERIFICATION: INSPECTING ALL PIPELINE SHEET SHAPES
 -> Weather (Dynamic Timeline):   (18260, 12) (Expected: 18260 rows)
 -> Crowd (Dynamic Timeline):     (18260, 17) (Expected: 18260 rows)
 -> Landslide (Static Dimension): (10, 13) (Expected: 10 rows)
 -> Transport (Static Dimension): (10, 10) (Expected: 10 rows)


## Executing the Sequential Merge & Row Guards

In [5]:
# Step 3: Merge Weather + Crowd on shared keys [location, date]
master = weather.merge(crowd, on=["location", "date"], how="left")

# ROW GUARD CHECK: Stop immediately if duplicate rows leaked into the timeline
if len(master) == 18260:
    print(f" ROW GUARD PASSED: Time-series join successful. Shape matches exactly: {master.shape}")
else:
    print(f" ROW GUARD FAILED: Duplicate keys detected! Length expanded to {len(master)}")
    print("Stopping pipeline execution to protect dataset integrity.")
    raise ValueError("Chronological timeline row distortion detected.")

 ROW GUARD PASSED: Time-series join successful. Shape matches exactly: (18260, 27)


In [6]:
# Step 4: Merge Landslide (Destination-level dimension data)
master = master.merge(landslide, on="location", how="left")
print(f" -> Landslide metrics broadcasted cleanly. Shape: {master.shape}")

 -> Landslide metrics broadcasted cleanly. Shape: (18260, 39)


In [7]:
# Step 5: Merge Transport (Destination-level dimension data)
master = master.merge(transport, on="location", how="left")
print(f" -> Transport metrics broadcasted cleanly. Shape: {master.shape}")

 -> Transport metrics broadcasted cleanly. Shape: (18260, 48)


## Structural Suffix De-duplication & Risk Score Placeholder

In [8]:
print("--- RUNNING PRODUCTION Suffix DE-DUPLICATION ENGINE ---")

# 1. Standardize overlapping column splits: drop the right-side (_y) duplicates completely
drop_cols = [c for c in master.columns if c.endswith('_y')]
master = master.drop(columns=drop_cols)

# 2. Clean up the left-side (_x) columns by stripping out their suffix names automatically
master.columns = [
    c.replace('_x', '') if c.endswith('_x') else c
    for c in master.columns
]

--- RUNNING PRODUCTION Suffix DE-DUPLICATION ENGINE ---


In [9]:
# 3. Create Final Trip Risk Score Target Placeholder for Notebook 05B/06
master["trip_risk_score"] = np.nan

print(" Schema de-duplicated safely. Primary columns preserved without manual re-injection.")

 Schema de-duplicated safely. Primary columns preserved without manual re-injection.


## Mandated Integrity Gate Checks & Sheet Export

In [10]:
# Save the clean master feature matrix directly to disk
output_filename = "master_feature_table.csv"
master.to_csv(output_filename, index=False)

In [11]:
print("=====================================================================")
print(f"🎉 TASK 05A COMPLETE: Master Feature Table written to '{output_filename}'")
print("=====================================================================\n")

print("=====================================================================")
print("🏁 FINAL INTEGRITY GATES REPORT")
print("=====================================================================")

🎉 TASK 05A COMPLETE: Master Feature Table written to 'master_feature_table.csv'

🏁 FINAL INTEGRITY GATES REPORT


In [12]:
# Check 1: Detailed Missing Values Report
print("1. Missing Values Report (Isolating non-placeholder nulls):")
null_report = master.drop(columns=['trip_risk_score']).isnull().sum()
print(null_report[null_report > 0])

1. Missing Values Report (Isolating non-placeholder nulls):
holiday_name    17470
dtype: int64


In [13]:
# Check 2: Timeline Key Duplicate Check
duplicates = master.duplicated(subset=["location", "date"]).sum()
print(f"\n2. Duplicate Keys Detected: {duplicates} (Expected: 0)")


2. Duplicate Keys Detected: 0 (Expected: 0)


In [14]:
# Check 3: Geographic Distribution Row Count Coverage
print("\n3. Row Count Coverage Breakdown per Destination:")
print(master.groupby("location").size())


3. Row Count Coverage Breakdown per Destination:
location
Darjeeling    1826
Goa           1826
Jaipur        1826
Leh           1826
Manali        1826
Munnar        1826
Mussoorie     1826
Nainital      1826
Ooty          1826
Shimla        1826
dtype: int64


In [15]:
if duplicates == 0 and len(master) == 18260:
    print("\n✅ ALL INTEGRITY GATES PASSED: Dataset is structurally perfect and ready for Notebook 05B!")
else:
    print("\n❌ INTEGRITY GATES FAILED: Please check for chronological merge distortions.")


✅ ALL INTEGRITY GATES PASSED: Dataset is structurally perfect and ready for Notebook 05B!
